# 02 — Feature Engineering — House Price Predictor

**Input:** `data/raw/train.csv`  
**Output:**
- `data/processed/X_train.csv`
- `data/processed/X_test.csv`
- `data/processed/y_train.csv`
- `data/processed/y_test.csv`
- `models/scaler.pkl`

## Decisions from EDA
- Drop: Id column
- Remove: 2 GrLivArea outliers (>4000 sqft, low price — confirmed errors)
- Target: log1p(SalePrice) — skew 1.88 → 0.12
- Structural NA: PoolQC, Alley, Fence, MiscFeature, FireplaceQu,
  all Bsmt* and Garage* → fill with 'None'/0
- Random missing: LotFrontage → neighborhood median imputation
- Ordinal encode: all quality/condition columns
- Target encode: Neighborhood (25 categories)
- One-hot encode: remaining low-cardinality categoricals
- Log-transform: skewed numeric features (|skew| > 0.75)
- New features: TotalSF, Total_Bathrooms, HouseAge, etc.

## RULE: Everything fit on X_train only. Never fit on X_test.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from scipy.stats import skew
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from src.data import load_raw_data, save_processed
from config import (TRAIN_FILE, PROCESSED_DATA_PATH, MODELS_PATH,
                    TARGET, ID_COL, RANDOM_SEED, TEST_SIZE, FIGURES_PATH)

pd.set_option('display.max_columns', 100)
print("✅ Imports OK")

In [ ]:
df = load_raw_data()
print(f"Raw shape: {df.shape}")
df.head(3)

In [ ]:
# Save IDs before dropping in case you need them later
ids = df[ID_COL].copy()

df = df.drop(columns=[ID_COL])
print(f"After dropping Id: {df.shape}")


In [ ]:
# Two confirmed data errors — large houses with suspiciously low prices
# Flagged in EDA, confirmed by dataset author Dean De Cock
before = len(df)
outlier_mask = (df['GrLivArea'] > 4000) & (df[TARGET] < 300000)
print(f"Outlier rows to remove: {outlier_mask.sum()}")
print(df[outlier_mask][['GrLivArea', TARGET]])

df = df[~outlier_mask].reset_index(drop=True)
print(f"\nRows before: {before}  After: {len(df)}  Removed: {before - len(df)}")

In [ ]:
# MSSubClass is stored as integer but is a categorical code
# MoSold is month number — treat as categorical
# YrSold — keep numeric (used in age calculations below)

ordinal_as_str = ['MSSubClass', 'MoSold']
df[ordinal_as_str] = df[ordinal_as_str].astype(str)

print("Fixed dtypes:")
print(df[ordinal_as_str].dtypes)

In [ ]:
# This MUST happen before any fitting of transformers
# Never let test set statistics influence training transformations

y = np.log1p(df[TARGET])    # log-transform the target
X = df.drop(columns=[TARGET])

print(f"X: {X.shape}")
print(f"y skew: {y.skew():.3f}  (was {df[TARGET].skew():.3f} before log)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED)

print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
# ── Columns where NA means the feature does not exist ─────────────────────────
none_cols_str = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]
none_cols_num = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt',
    'GarageCars', 'GarageArea', 'MasVnrArea'
]

for col in none_cols_str:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('None')
        X_test[col]  = X_test[col].fillna('None')

for col in none_cols_num:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna(0)
        X_test[col]  = X_test[col].fillna(0)

print("Structural NA handled.")
print(f"Missing after structural fix:")
remaining = X_train.isnull().sum()
print(remaining[remaining > 0])

In [ ]:
# ── LotFrontage: impute with median per Neighborhood ──────────────────────────
# Houses in the same neighborhood tend to have similar lot frontages
# Much smarter than global median

lot_median_by_hood = X_train.groupby('Neighborhood')['LotFrontage'].median()

def impute_lot_frontage(df, mapping):
    df['LotFrontage'] = df.apply(
        lambda row: mapping.get(row['Neighborhood'], X_train['LotFrontage'].median())
        if pd.isnull(row['LotFrontage']) else row['LotFrontage'],
        axis=1
    )
    return df

X_train = impute_lot_frontage(X_train, lot_median_by_hood)
X_test  = impute_lot_frontage(X_test,  lot_median_by_hood)

print(f"LotFrontage missing after imputation: {X_train['LotFrontage'].isnull().sum()}")

In [ ]:
# ── Remaining random missing: fit imputers on train, apply to both ─────────────
remaining_num_missing = (X_train.select_dtypes(include='number')
                                 .isnull().sum())
remaining_num_missing = remaining_num_missing[remaining_num_missing > 0].index.tolist()
print(f"Remaining numeric missing: {remaining_num_missing}")

if remaining_num_missing:
    num_imputer = SimpleImputer(strategy='median')
    X_train[remaining_num_missing] = num_imputer.fit_transform(
        X_train[remaining_num_missing])
    X_test[remaining_num_missing]  = num_imputer.transform(
        X_test[remaining_num_missing])

remaining_cat_missing = (X_train.select_dtypes(include='object')
                                  .isnull().sum())
remaining_cat_missing = remaining_cat_missing[remaining_cat_missing > 0].index.tolist()
print(f"Remaining categorical missing: {remaining_cat_missing}")

if remaining_cat_missing:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X_train[remaining_cat_missing] = cat_imputer.fit_transform(
        X_train[remaining_cat_missing])
    X_test[remaining_cat_missing]  = cat_imputer.transform(
        X_test[remaining_cat_missing])

# Final verification
assert X_train.isnull().sum().sum() == 0, "Missing values remain in X_train"
assert X_test.isnull().sum().sum()  == 0, "Missing values remain in X_test"
print("✅ Zero missing values in both sets")

In [ ]:
# Do this BEFORE encoding — new features use the original columns

for df_part in [X_train, X_test]:

    # ── Additive combinations ─────────────────────────────────────────────────
    df_part['TotalSF'] = (df_part['TotalBsmtSF']
                           + df_part['1stFlrSF']
                           + df_part['2ndFlrSF'])

    df_part['Total_Bathrooms'] = (df_part['FullBath']
                                   + 0.5 * df_part['HalfBath']
                                   + df_part['BsmtFullBath']
                                   + 0.5 * df_part['BsmtHalfBath'])

    df_part['Total_Porch_SF'] = (df_part['OpenPorchSF']
                                  + df_part['EnclosedPorch']
                                  + df_part['3SsnPorch']
                                  + df_part['ScreenPorch'])

    # ── Multiplicative interactions ───────────────────────────────────────────
    # Quality × size = more informative than either alone
    df_part['Quality_x_Area']    = df_part['OverallQual'] * df_part['GrLivArea']
    df_part['Quality_x_TotalSF'] = df_part['OverallQual'] * df_part['TotalSF']

    # ── Binary flags ──────────────────────────────────────────────────────────
    df_part['HasPool']      = (df_part['PoolArea']    > 0).astype(int)
    df_part['HasGarage']    = (df_part['GarageArea']  > 0).astype(int)
    df_part['HasBasement']  = (df_part['TotalBsmtSF'] > 0).astype(int)
    df_part['HasFireplace'] = (df_part['Fireplaces']  > 0).astype(int)
    df_part['Has2ndFloor']  = (df_part['2ndFlrSF']    > 0).astype(int)
    df_part['HasPorch']     = (df_part['Total_Porch_SF'] > 0).astype(int)

    # ── Time features ─────────────────────────────────────────────────────────
    df_part['HouseAge']    = df_part['YrSold'] - df_part['YearBuilt']
    df_part['RemodAge']    = df_part['YrSold'] - df_part['YearRemodAdd']
    df_part['IsRemodeled'] = (df_part['YearBuilt'] != df_part['YearRemodAdd']).astype(int)
    df_part['IsNewHouse']  = (df_part['YearBuilt'] == df_part['YrSold']).astype(int)

    # GarageYrBlt was filled with 0 for no-garage houses
    # Avoid negative ages from that
    df_part['GarageAge'] = df_part['YrSold'] - df_part['GarageYrBlt']
    df_part['GarageAge'] = df_part['GarageAge'].clip(lower=0)

print(f"X_train shape after feature creation: {X_train.shape}")
print(f"New features added: {X_train.shape[1] - X.shape[1]}")

In [ ]:
# ── Neighborhood group statistics — fit on train, map to both ─────────────────
# "How does this house compare to its neighborhood average?"

hood_stats = (X_train.join(y_train)
                      .groupby('Neighborhood')[TARGET]
                      .agg(Neighborhood_MeanPrice='mean',
                           Neighborhood_MedianPrice='median',
                           Neighborhood_StdPrice='std')
                      .fillna(0))

X_train = X_train.join(hood_stats, on='Neighborhood')
X_test  = X_test.join(hood_stats, on='Neighborhood')

print(f"Shape after neighborhood stats: {X_train.shape}")

In [ ]:
# ── Ordinal encoding — quality and condition columns ──────────────────────────
# These have a genuine order: Poor < Fair < Typical < Good < Excellent
# Preserving order matters — do NOT one-hot encode these

quality_map    = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'None': 0}
exposure_map   = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
finish_map     = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3,
                   'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
garage_fin_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
lot_shape_map  = {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4}
slope_map      = {'Sev': 1, 'Mod': 2, 'Gtl': 3}
paved_map      = {'N': 0, 'P': 1, 'Y': 2}
fence_map      = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}

ordinal_encodings = {
    'ExterQual':    quality_map,
    'ExterCond':    quality_map,
    'BsmtQual':     quality_map,
    'BsmtCond':     quality_map,
    'HeatingQC':    quality_map,
    'KitchenQual':  quality_map,
    'FireplaceQu':  quality_map,
    'GarageQual':   quality_map,
    'GarageCond':   quality_map,
    'PoolQC':       quality_map,
    'BsmtExposure': exposure_map,
    'BsmtFinType1': finish_map,
    'BsmtFinType2': finish_map,
    'GarageFinish': garage_fin_map,
    'LotShape':     lot_shape_map,
    'LandSlope':    slope_map,
    'PavedDrive':   paved_map,
    'Fence':        fence_map,
}

for col, mapping in ordinal_encodings.items():
    if col in X_train.columns:
        X_train[col] = X_train[col].map(mapping).fillna(0)
        X_test[col]  = X_test[col].map(mapping).fillna(0)

print(f"Ordinal encoded {len(ordinal_encodings)} columns")

In [ ]:
# ── Target encoding — Neighborhood (25 unique values) ─────────────────────────
# Replaces each neighborhood with the smoothed mean SalePrice of that neighborhood
# Fit on train only — map to test using train statistics

smoothing = 10
global_mean = y_train.mean()

hood_target_stats = (X_train.assign(y=y_train.values)
                             .groupby('Neighborhood')['y']
                             .agg(['mean', 'count']))

hood_target_stats['smooth'] = (
    (hood_target_stats['count'] * hood_target_stats['mean']
     + smoothing * global_mean)
    / (hood_target_stats['count'] + smoothing)
)

mapping = hood_target_stats['smooth'].to_dict()

X_train['Neighborhood_enc'] = X_train['Neighborhood'].map(mapping).fillna(global_mean)
X_test['Neighborhood_enc']  = X_test['Neighborhood'].map(mapping).fillna(global_mean)

# Drop original Neighborhood — information is now in Neighborhood_enc
X_train = X_train.drop(columns=['Neighborhood'])
X_test  = X_test.drop(columns=['Neighborhood'])

print(f"Neighborhood target-encoded. Sample:")
print(X_train['Neighborhood_enc'].describe())

In [ ]:
# ── One-hot encoding — remaining low-cardinality nominals ─────────────────────
# Nominal = no order (MSZoning, SaleType, Foundation...)
# Only encode columns with <= 10 unique values to control feature count

remaining_cat = X_train.select_dtypes(include='object').columns.tolist()
low_card   = [c for c in remaining_cat if X_train[c].nunique() <= 10]
high_card  = [c for c in remaining_cat if X_train[c].nunique() >  10]

print(f"One-hot encoding {len(low_card)} columns: {low_card}")
print(f"\nHigh cardinality still present (will label encode): {high_card}")

X_train = pd.get_dummies(X_train, columns=low_card, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=low_card, drop_first=True)

# Align — test may be missing dummies present in train (rare categories)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"\nShape after one-hot encoding:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
# ── Label encode any remaining high-cardinality categoricals ──────────────────
from sklearn.preprocessing import LabelEncoder

remaining_cat_final = X_train.select_dtypes(include='object').columns.tolist()
print(f"Remaining categoricals to label encode: {remaining_cat_final}")

for col in remaining_cat_final:
    le = LabelEncoder()
    le.fit(X_train[col].astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))

    # Handle unseen categories in test gracefully
    known = set(le.classes_)
    X_test[col] = X_test[col].astype(str).apply(
        lambda x: x if x in known else le.classes_[0])
    X_test[col] = le.transform(X_test[col])

print("✅ All categorical columns encoded")
print(f"\nRemaining object columns: {X_train.select_dtypes(include='object').columns.tolist()}")

In [ ]:
# Compute skew on training set only
numeric_feats = X_train.select_dtypes(include='number').columns

skewness = X_train[numeric_feats].apply(lambda x: skew(x.dropna()))
skewness = skewness.sort_values(ascending=False)

high_skew = skewness[abs(skewness) > 0.75].index.tolist()

# Don't transform binary (0/1) or ordinal (already small integers)
binary_cols  = [c for c in high_skew if X_train[c].nunique() <= 2]
skip_cols    = binary_cols
high_skew    = [c for c in high_skew if c not in skip_cols]

print(f"Log-transforming {len(high_skew)} skewed features:")
print(high_skew[:10], "..." if len(high_skew) > 10 else "")

for col in high_skew:
    X_train[col] = np.log1p(X_train[col].clip(lower=0))
    X_test[col]  = np.log1p(X_test[col].clip(lower=0))

print("\n✅ Skew transformation complete")
print(f"Skew after transform (sample):")
print(X_train[high_skew[:5]].apply(lambda x: skew(x)).round(3))

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train)

low_var = X_train.columns[~selector.get_support()].tolist()
print(f"Low variance columns to remove: {low_var}")

if low_var:
    X_train = X_train.drop(columns=low_var)
    X_test  = X_test.drop(columns=low_var)

print(f"Shape after variance filter: {X_train.shape}")

In [ ]:
# RobustScaler uses median and IQR — less sensitive to outliers
# Fit on train only, transform both

numeric_cols_final = X_train.select_dtypes(include='number').columns.tolist()

scaler = RobustScaler()
X_train[numeric_cols_final] = scaler.fit_transform(X_train[numeric_cols_final])
X_test[numeric_cols_final]  = scaler.transform(X_test[numeric_cols_final])

# Save scaler — needed at prediction time to transform new data
joblib.dump(scaler, MODELS_PATH / 'scaler.pkl')
print(f"✅ Scaler fitted and saved")
print(f"Scaled {len(numeric_cols_final)} numeric columns")

In [ ]:
#final verification
print("═" * 50)
print("  FINAL VERIFICATION")
print("═" * 50)
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")
print(f"\nMissing X_train: {X_train.isnull().sum().sum()}")
print(f"Missing X_test:  {X_test.isnull().sum().sum()}")
print(f"Missing y_train: {y_train.isnull().sum()}")
print(f"\nDtype breakdown:")
print(X_train.dtypes.value_counts())

# Shape must match
assert X_train.shape[1] == X_test.shape[1],  "Column mismatch!"
assert X_train.shape[0] == y_train.shape[0], "Row mismatch train!"
assert X_test.shape[0]  == y_test.shape[0],  "Row mismatch test!"
assert X_train.isnull().sum().sum() == 0,     "Missing values in train!"
assert X_test.isnull().sum().sum()  == 0,     "Missing values in test!"

print("\n✅ All assertions passed")

In [ ]:
X_train.to_csv(PROCESSED_DATA_PATH / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DATA_PATH  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DATA_PATH / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DATA_PATH  / 'y_test.csv',  index=False)

print("✅ Saved to data/processed/:")
for f in sorted(PROCESSED_DATA_PATH.iterdir()):
    print(f"  {f.name:<25} {f.stat().st_size / 1024:.1f} KB")

## Feature Engineering Summary

### Changes Made
- Dropped: Id
- Removed: 2 GrLivArea outliers (confirmed data errors)
- Target: log1p(SalePrice) — skew 1.88 → 0.12
- Structural NA → 'None'/0 for: Alley, BsmtQual, BsmtCond, BsmtExposure,
  BsmtFinType1/2, FireplaceQu, GarageType/Finish/Qual/Cond, GarageYrBlt,
  GarageCars, GarageArea, PoolQC, Fence, MiscFeature, MasVnrType, MasVnrArea
- LotFrontage: neighborhood-median imputation
- MSSubClass, MoSold: converted to string (categorical codes)
- Created 16 new features: TotalSF, Total_Bathrooms, Total_Porch_SF,
  Quality_x_Area, Quality_x_TotalSF, HasPool, HasGarage, HasBasement,
  HasFireplace, Has2ndFloor, HasPorch, HouseAge, RemodAge,
  IsRemodeled, IsNewHouse, GarageAge
- Neighborhood group stats: Neighborhood_MeanPrice, MedianPrice, StdPrice
- Ordinal encoded: 18 quality/condition columns (order preserved)
- Target encoded: Neighborhood (smoothed, train-only fit)
- One-hot encoded: all low-cardinality (<= 10 unique) nominal columns
- Log-transformed: all numeric features with |skew| > 0.75
- Removed low-variance features
- RobustScaler applied: fit on train, transform both

### Final Dataset
- X_train: 1,166 rows × [N] features
- X_test:  292  rows × [N] features
